In [36]:
import pandas as pd
import numpy as np

In [37]:
df = pd.read_csv('../data/processed/restaurant_features.csv')
print(df.shape)

(9542, 134)


In [38]:
y = df['Aggregate rating']
drop_cols = [
    'Aggregate rating',
    'Restaurant ID',
    'Restaurant Name',
    'Address',
    'Locality',
    'Locality Verbose',
    'Rating color',
    'Rating text',
    'Cuisine_List',
    'Cuisines',
    'Currency'
]

X = df.drop(columns=drop_cols, errors='ignore')

In [39]:
print("X shape:", X.shape)
print("\nData types:")
print(X.dtypes.value_counts())

print("\nRemaining object columns:")
print(X.select_dtypes(include='object').columns.tolist())

X shape: (9542, 130)

Data types:
int64      123
float64      7
Name: count, dtype: int64

Remaining object columns:
[]


In [40]:
correlation = (
    X.assign(**{'Aggregate rating': y})
     .corr(numeric_only=True)['Aggregate rating']
     .sort_values(ascending=False)
)

print(correlation)

Aggregate rating                 1.000000
Log_Votes                        0.848323
Price range                      0.438356
Log_Relative_Cost_by_Currency    0.375983
Votes                            0.313474
                                   ...   
Cuisine_Street Food             -0.051004
Cuisine_Mithai                  -0.079441
Cuisine_North Indian            -0.085967
Longitude                       -0.114733
Currency_Indian Rupees(Rs.)     -0.291772
Name: Aggregate rating, Length: 131, dtype: float64


In [41]:
correlation = correlation.drop('Aggregate rating')

print(correlation)

Log_Votes                        0.848323
Price range                      0.438356
Log_Relative_Cost_by_Currency    0.375983
Votes                            0.313474
Relative_Cost_by_Currency        0.303201
                                   ...   
Cuisine_Street Food             -0.051004
Cuisine_Mithai                  -0.079441
Cuisine_North Indian            -0.085967
Longitude                       -0.114733
Currency_Indian Rupees(Rs.)     -0.291772
Name: Aggregate rating, Length: 130, dtype: float64


In [42]:

# Check for features that are strongly correlated with each other
feature_corr = X.corr(numeric_only=True)

threshold = 0.85
high_corr = []

columns = feature_corr.columns

for i in range(len(columns)):
    for j in range(i):
        value = feature_corr.iloc[i, j]

        if abs(value) >= threshold:
            high_corr.append({
                'Feature 1': columns[i],
                'Feature 2': columns[j],
                'Correlation': round(value, 4)
            })

high_corr_df = pd.DataFrame(high_corr)

if high_corr_df.empty:
    print("No highly correlated feature pairs found.")
else:
    print(
        high_corr_df.sort_values(
            by='Correlation',
            key=lambda s: s.abs(),
            ascending=False
        ).to_string(index=False)
    )


                    Feature 1                 Feature 2  Correlation
         Cost_per_Price_Range      Average Cost for two       0.9995
Log_Relative_Cost_by_Currency Relative_Cost_by_Currency       0.9304


In [43]:
X=X.drop(columns=["Cost_Per_Price_Range"],errors='ignore')
print("Updated",X.shape)

Updated (9542, 130)


In [44]:
candidate_pairs = [
    ('Votes', 'Log_Votes'),
    ('Average Cost for two', 'Log_Average_Cost'),
    ('Relative_Cost_by_Currency',
     'Log_Relative_Cost_by_Currency'),
    ('Has Table booking', 'Service_Count'),
    ('Has Online delivery', 'Service_Count'),
    ('Is delivering now', 'Service_Count'),
    ('Cuisine_Count', 'Has_Multiple_Cuisines')
]

for col1, col2 in candidate_pairs:
    if col1 in X.columns and col2 in X.columns:
        value = X[col1].corr(X[col2])
        print(f"{col1} ↔ {col2}: {value:.4f}")
    else:
        print(f"Missing pair: {col1}, {col2}")

Votes ↔ Log_Votes: 0.5512
Average Cost for two ↔ Log_Average_Cost: 0.2678
Relative_Cost_by_Currency ↔ Log_Relative_Cost_by_Currency: 0.9304
Has Table booking ↔ Service_Count: 0.6392
Has Online delivery ↔ Service_Count: 0.8229
Is delivering now ↔ Service_Count: 0.1703
Cuisine_Count ↔ Has_Multiple_Cuisines: 0.7246


In [45]:
# checking existence 

In [46]:
columns_check=[('Cuisine_Count'),('Has_Multiple_Cuisines')]
for columns in columns_check:
    if columns in df.columns:
        print(f"Present,{columns}")
    else:
        print(f"Missing,{columns}")

for columns in columns_check:
    if columns in X.columns:
        print(f"Yes,{columns}")
    else:
        print(f"No,{columns}")

        

Present,Cuisine_Count
Present,Has_Multiple_Cuisines
Yes,Cuisine_Count
Yes,Has_Multiple_Cuisines


In [47]:
columns_check=[('Cuisine_List'),('Cuisines')]
for columns in columns_check:
    if columns in df.columns:
        print(f"Present,{columns}")
    else:
        print(f"Missing,{columns}")

for columns in columns_check:
    if columns in X.columns:
        print(f"Yes,{columns}")
    else:
        print(f"No,{columns}")

        

Present,Cuisine_List
Present,Cuisines
No,Cuisine_List
No,Cuisines


In [48]:
df['Cuisine_Count'].isna().sum()
df['Cuisine_Count'].describe()


count    9542.000000
mean        2.065605
std         1.092750
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         8.000000
Name: Cuisine_Count, dtype: float64

In [49]:
non_strings_in_col1 = df['Cuisine_Count'].apply(lambda x: isinstance(x, str))
print(df[non_strings_in_col1])

# df[some_column] == some_value 
# df[some_column].isin(some_list_of_values) # This check whether the value of the column is one of the values in the list
# df[some_column].str.contains() # You can use it the same as str.contains()
# df[some_column].str.isdigit() # Same usage as str.isdigit(), check whether string is all digits, need to make sure column type is string in advance
# df[some_column].str.len() == 4 # Filter string with length of 4


Empty DataFrame
Columns: [Longitude, Latitude, Cuisines, Average Cost for two, Currency, Has Table booking, Has Online delivery, Is delivering now, Price range, Aggregate rating, Votes, Cuisine_List, Has_Multiple_Cuisines, Cuisine_Count, Cuisine_Afghani, Cuisine_African, Cuisine_American, Cuisine_Andhra, Cuisine_Arabian, Cuisine_Asian, Cuisine_Australian, Cuisine_Awadhi, Cuisine_BBQ, Cuisine_Bakery, Cuisine_Bar Food, Cuisine_Bengali, Cuisine_Beverages, Cuisine_Bihari, Cuisine_Biryani, Cuisine_Brazilian, Cuisine_Breakfast, Cuisine_British, Cuisine_Burger, Cuisine_Burmese, Cuisine_Cafe, Cuisine_Cajun, Cuisine_Caribbean, Cuisine_Chettinad, Cuisine_Chinese, Cuisine_Coffee and Tea, Cuisine_Contemporary, Cuisine_Continental, Cuisine_Curry, Cuisine_Desserts, Cuisine_Diner, Cuisine_European, Cuisine_Fast Food, Cuisine_Filipino, Cuisine_Finger Food, Cuisine_French, Cuisine_German, Cuisine_Goan, Cuisine_Greek, Cuisine_Grill, Cuisine_Gujarati, Cuisine_Hawaiian, Cuisine_Healthy Food, Cuisine_Hyder

In [50]:
check_cols= df[['Cuisine_List','Cuisine_Count']]
def verify_rel(row):
  if isinstance(row['Cuisine_List'],list):
    return len(row['Cuisine_List'])== row['Cuisine_Count']
  return False

df['Count_Match']= check_cols.apply(verify_rel,axis=1)
all_match= df['Count_Match'].all()

print(df)
if all_match:
    print(" All Cuisine_Count values match the length of Cuisine_List.")
else:
    print("Mismatch found in some rows.")
    print(df.loc[~df['Count_Match'], ['Cuisine_List', 'Cuisine_Count']])


       Longitude   Latitude                          Cuisines  \
0     121.027535  14.565443        French, Japanese, Desserts   
1     121.014101  14.553708                          Japanese   
2     121.056831  14.581404  Seafood, Asian, Filipino, Indian   
3     121.056475  14.585318                   Japanese, Sushi   
4     121.057508  14.584450                  Japanese, Korean   
...          ...        ...                               ...   
9537   28.977392  41.022793                           Turkish   
9538   29.041297  41.009847   World Cuisine, Patisserie, Cafe   
9539   29.034640  41.055817            Italian, World Cuisine   
9540   29.036019  41.057979                   Restaurant Cafe   
9541   29.026016  40.984776                              Cafe   

      Average Cost for two          Currency  Has Table booking  \
0                     1100  Botswana Pula(P)                  1   
1                     1200  Botswana Pula(P)                  1   
2                 

In [51]:
def row_check(row):
    if len(row['Cuisine_List']) == row['Cuisine_Count']:
        return True
    else:
        return False

print("Cuisine list:", df['Cuisine_List'].dtype)
print("Cuisine count:", df['Cuisine_Count'].dtype)

# Check the first row
print("Row 0 matches:", row_check(df.iloc[0]))


Cuisine list: object
Cuisine count: int64
Row 0 matches: False


In [52]:
df['Count_Match'] = df.apply(row_check, axis=1)

print("Matching rows:", df['Count_Match'].sum())
print("Mismatched rows:", (~df['Count_Match']).sum())


Matching rows: 0
Mismatched rows: 9542


In [53]:

row = df.iloc[0]

print("Cuisine_List:", repr(row['Cuisine_List']))
print("List type:", type(row['Cuisine_List']))
print("Cuisine_Count:", repr(row['Cuisine_Count']))
print("Count type:", type(row['Cuisine_Count']))
print("List length:", len(row['Cuisine_List']))
print("Direct comparison:", len(row['Cuisine_List']) == row['Cuisine_Count'])


Cuisine_List: "['French', 'Japanese', 'Desserts']"
List type: <class 'str'>
Cuisine_Count: np.int64(3)
Count type: <class 'numpy.int64'>
List length: 34
Direct comparison: False


In [54]:
print("Cuisine list sample",df['Cuisine_List'].head(5).tolist())
print("Cuisine List ",df['Cuisine_List'].iloc[0])
print("Cuisine count sample",df['Cuisine_Count'].head(5).tolist())
print("Cusine Count",df['Cuisine_Count'].iloc[0])

Cuisine list sample ["['French', 'Japanese', 'Desserts']", "['Japanese']", "['Seafood', 'Asian', 'Filipino', 'Indian']", "['Japanese', 'Sushi']", "['Japanese', 'Korean']"]
Cuisine List  ['French', 'Japanese', 'Desserts']
Cuisine count sample [3, 1, 4, 2, 2]
Cusine Count 3


In [55]:
import ast 
df['Cuisine_List']= df['Cuisine_List'].apply(ast.literal_eval)

print("Element type",type(df['Cuisine_List'].iloc[0]))
print("Cuisine list:", df["Cuisine_List"].iloc[0])
print("Cuisine count:", df["Cuisine_Count"].iloc[0])
      

Element type <class 'list'>
Cuisine list: ['French', 'Japanese', 'Desserts']
Cuisine count: 3


In [56]:

df["Count_Match"] = df.apply(
    lambda row: len(row["Cuisine_List"]) == row["Cuisine_Count"],
    axis=1
)

print("Matching rows:", df["Count_Match"].sum())
print("Mismatched rows:", (~df["Count_Match"]).sum())


Matching rows: 9542
Mismatched rows: 0


In [57]:

# 1. Missing values
missing_cols = df.columns[df.isna().any()].tolist()

# 2. Constant columns
list_cols = [
    col for col in df.columns
    if df[col].apply(lambda x: isinstance(x, list)).any()
]

constant_cols = [
    col for col in df.columns
    if col not in list_cols
    and df[col].nunique(dropna=False) <= 1
]

# 3. Duplicate columns, excluding list-valued columns
check_df = df.drop(columns=list_cols)
duplicate_cols = check_df.columns[
    check_df.T.duplicated()
].tolist()

print("Columns with missing values:", missing_cols)
print("Constant columns:", constant_cols)
print("Duplicate columns:", duplicate_cols)


Columns with missing values: []
Constant columns: ['Count_Match']
Duplicate columns: []


In [58]:
df.drop(columns=['Count_Match'], inplace=True)

In [59]:
if 'Aggregate rating' in X.columns:
    print(True)
else:
    print(False)

False


In [60]:
# Example
Y = df['Aggregate rating']
print("Features shape:", X.shape)
print("Target shape:", Y.shape)

Features shape: (9542, 130)
Target shape: (9542,)


In [61]:
missing = X.isna().sum()
print(missing[missing > 0])

Series([], dtype: int64)


In [62]:
# Example: checking for unwanted columns
columns_to_check = ["Count_Match", "Restaurant ID","Restaurant Name","Address", "Locality","Locality Verbose","Rating color", "Rating text","Cuisines","Cuisine_List","Currency" ,"Location","Country Code", "City", "Latitude", "Longitude",
"Switch to order menu", "Aggregate rating"]

found = [col for col in columns_to_check if col in X.columns]

print(found)

['Latitude', 'Longitude']


In [63]:
X.drop(columns=['Latitude','Longitude'],inplace=True)



In [64]:
if 'Latitude' in X.columns and 'Longitude' in X.columns:
    print(True)
else:
    print(False)

False


In [65]:
if 'Aggregate rating' in X.columns:
    print(True)
else:
    print(False)

False


In [68]:
if 'Aggregate rating' in Y:
    print(True)
else:
    print(Y.name)

Aggregate rating


In [70]:
type(Y)

pandas.core.series.Series

In [71]:
print(X.shape)
print(Y.shape)

print(X.index.equals(Y.index))

(9542, 128)
(9542,)
True


In [75]:
cols=['Rating color','Rating text']

for cols in cols:
    if cols in X.columns:
        print(True)
    else:
        print(False)
      

False
False


In [79]:
print(X[['Votes','Log_Votes']].describe())
print(X[['Votes','Log_Votes']].skew())

              Votes    Log_Votes
count   9542.000000  9542.000000
mean     156.772060     3.329798
std      430.203324     2.036994
min        0.000000     0.000000
25%        5.000000     1.791759
50%       31.000000     3.465736
75%      130.000000     4.875197
max    10934.000000     9.299724
Votes        8.813270
Log_Votes   -0.028783
dtype: float64


In [80]:
X[['Votes','Log_Votes']].head(10)

,Votes,Log_Votes
0,314,5.752573
1,591,6.383507
2,270,5.602119
3,365,5.902633
4,229,5.438079
5,336,5.820083
6,520,6.255750
7,677,6.519147
8,621,6.432940
9,532,6.278521


In [81]:
X[['Votes','Log_Votes']].corr()

,Votes,Log_Votes
Votes,1.000000,0.551155
Log_Votes,0.551155,1.000000


In [82]:
from scipy.stats import spearmanr

f1=df['Votes']
f2=df['Log_Votes']

rho, pval = spearmanr(f1, f2)
print(rho, pval)



0.9999999999999999 0.0
